In [ ]:
import hashlib
import os
import random
import sys
import time
from urllib.parse import quote
import json
import requests

In [ ]:
ZENODO_SANDBOX_TOKEN = 'add_sandbox_token_here'
ZENODO_TOKEN = 'add_token_here'

In [ ]:
# CONFIGURATION

# Use the sandbox until the script does what you want. Sandbox tokens are
# separate from production tokens, and sandbox DOIs are throwaway.
SANDBOX = False

# Existing unpublished draft to upload into, e.g. 1234567 taken from the URL
# of the draft in the web interface (.../uploads/1234567). Leave as None to
# create a fresh empty deposition instead.
DEPOSITION_ID = None # add deposition ID here

# Files to upload. They are stored under their basename in the record.
FILES = [
    # add file paths here
]

# Set to True only when you are sure: published records cannot be deleted,
# and files cannot be changed after publishing.
PUBLISH = False

# Leave False if you already filled in the metadata in the web interface.
# PUT replaces the whole metadata object rather than merging into it, so
# sending a partial METADATA below would wipe the fields you already entered.
SET_METADATA = False

# Deposition metadata. Required fields: title, upload_type, description,
# creators. See https://developers.zenodo.org/#representation for the rest.
METADATA = {
    # add metadata here
    }

# Attempts per file within a single run. Zenodo's bucket API cannot resume a
# broken PUT, so every attempt restarts that one file from byte 0 -- but a
# file that already finished is never re-sent, so progress is kept across
# both attempts and reruns. On a link that drops every few minutes, set this
# high and let it grind; nothing is lost by trying again.
MAX_ATTEMPTS = 25
BACKOFF_CAP = 120  # seconds; backoff doubles up to this, then stays flat

READ_CHUNK = 8 * 1024 * 1024  # only used for local checksumming

In [ ]:
# SETUP AND SANITY CHECKS

if SANDBOX:
    BASE_URL = "https://sandbox.zenodo.org/api"
    TOKEN = ZENODO_SANDBOX_TOKEN
    TOKEN_VAR = "ZENODO_SANDBOX_TOKEN"
else:
    BASE_URL = "https://zenodo.org/api"
    TOKEN = ZENODO_TOKEN
    TOKEN_VAR = "ZENODO_TOKEN"

if not TOKEN:
    sys.exit(f"No access token found. Set ${TOKEN_VAR} in your environment.")

for path in FILES:
    if not os.path.isfile(path):
        sys.exit(f"Not a file: {path}")

session = requests.Session()
# Token goes in a header, not the query string, so it does not end up in
# server logs or in your shell history.
session.headers["Authorization"] = f"Bearer {TOKEN}"

In [ ]:
# Where to remember local checksums between runs. Checksumming 12 GB takes
# minutes, and on a flaky link you will rerun this notebook many times, so
# the digest is cached against (size, mtime) and only recomputed if the file
# actually changed on disk.
MD5_CACHE_PATH = r"insert_md5_cache_path
# urllib3 retries a dropped connection by itself, but it cannot rewind a
# file object that has already been partly read, so its retry always fails
# with "Max retries exceeded". Turning it off makes the real error surface
# immediately and leaves retrying to the loop below, which reopens the file.
session.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))

def check(response):
    """Raise on HTTP errors, but print Zenodo's error body first."""
    if not response.ok:
        print(f"\nHTTP {response.status_code} for {response.request.method} "
              f"{response.url}\n{response.text}", file=sys.stderr)
    response.raise_for_status()
    return response


def _load_cache():
    try:
        with open(MD5_CACHE_PATH, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except (OSError, ValueError):
        return {}

def md5_of(path):
    """Streaming MD5, cached on (size, mtime) so reruns do not rehash 12 GB."""
    key = os.path.abspath(path)
    stat = os.stat(path)
    signature = [stat.st_size, int(stat.st_mtime)]

    cache = _load_cache()
    cached = cache.get(key)
    if cached and cached[:2] == signature:
        return cached[2]

    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(READ_CHUNK), b""):
            digest.update(block)
    result = digest.hexdigest()

    cache[key] = signature + [result]
    try:
        with open(MD5_CACHE_PATH, "w", encoding="utf-8") as handle:
            json.dump(cache, handle, indent=1)
    except OSError as error:
        print(f"  (could not write checksum cache: {error})", file=sys.stderr)
    return result


def remote_entries(deposition_id):
    """
    Map filename -> file entry for what the draft currently holds.

    This uses the newer records API rather than the legacy deposit API
    because it reports `status` and `checksum` per file, which is what lets
    a rerun tell a finished upload from a half-finished one.
    """
    response = check(session.get(
        f"{BASE_URL}/records/{deposition_id}/draft/files", timeout=60))
    return {entry["key"]: entry for entry in response.json().get("entries", [])}


class ProgressFile:
    """
    A read-only file wrapper that prints upload progress.

    __len__ matters: it lets requests set a Content-Length header instead of
    falling back to chunked transfer encoding, which the storage backend
    behind Zenodo does not accept for large bodies.
    """

    def __init__(self, path):
        self.handle = open(path, "rb")
        self.size = os.path.getsize(path)
        self.name = os.path.basename(path)
        self.sent = 0
        self.started = time.monotonic()
        self.last_print = 0.0

    def __len__(self):
        return self.size

    def read(self, size=-1):
        block = self.handle.read(size)
        self.sent += len(block)
        now = time.monotonic()
        if now - self.last_print > 0.5 or self.sent == self.size:
            self.last_print = now
            elapsed = max(now - self.started, 1e-6)
            pct = 100.0 * self.sent / self.size if self.size else 100.0
            mib = self.sent / 1024 / 1024
            rate = mib / elapsed
            print(f"\r  {self.name}: {pct:5.1f}%  {mib:9.1f} MiB  "
                  f"{rate:6.1f} MiB/s", end="", flush=True)
        return block

    def close(self):
        self.handle.close()


In [ ]:
# STEP 1: get the deposition to upload into, and its bucket URL

if DEPOSITION_ID is None:
    print("Creating draft deposition ...")
    response = check(session.post(f"{BASE_URL}/deposit/depositions",
                                  json={}, timeout=30))
else:
    print(f"Fetching draft deposition {DEPOSITION_ID} ...")
    response = check(session.get(
        f"{BASE_URL}/deposit/depositions/{DEPOSITION_ID}", timeout=30))

deposition = response.json()
deposition_id = deposition["id"]

# Files can only be added to an unpublished draft. A published record needs
# POST /actions/newversion first, which gives you a fresh draft to upload to.
if deposition.get("submitted"):
    sys.exit(f"Deposition {deposition_id} is already published; its files "
             f"cannot be changed. Create a new version instead.")

bucket_url = deposition["links"]["bucket"]
print(f"  title  : {deposition.get('title') or '(untitled)'}")
print(f"  state  : {deposition.get('state')}")
print(f"  files  : {len(deposition.get('files', []))} already in the record")
print(f"  bucket : {bucket_url}")

In [ ]:
# STEP 2: upload the files
# Nothing in here calls sys.exit(). A file that will not go through is
# recorded as a failure and the loop moves on to the next one, so one bad
# file no longer costs you the whole run. Rerun this cell as often as you
# like: files already sitting complete in the draft are skipped, so every
# run only has to deal with what is still missing.

def upload_file(path, already_there):
    """Upload one file. Returns True if it is complete in the draft."""
    filename = os.path.basename(path)
    size = os.path.getsize(path)
    size_mib = size / 1024 / 1024

    print(f"\n{filename} ({size_mib:.1f} MiB)")
    local_md5 = md5_of(path)

    entry = already_there.get(filename)
    if (entry
            and entry.get("status") == "completed"
            and entry.get("size") == size
            and entry.get("checksum") == f"md5:{local_md5}"):
        print("  already in the draft with a matching checksum, skipping")
        return True
    if entry:
        print(f"  present but not usable (status={entry.get('status')}, "
              f"size={entry.get('size')}), re-uploading")

    # A re-PUT of the same filename simply overwrites what is there, so an
    # interrupted upload leaves nothing to clean up first.
    target = f"{bucket_url}/{quote(filename)}"

    for attempt in range(1, MAX_ATTEMPTS + 1):
        print(f"  attempt {attempt}/{MAX_ATTEMPTS}")
        body = ProgressFile(path)
        try:
            # timeout is per socket operation, not for the whole transfer,
            # so a slow-but-progressing upload will not trip it.
            response = session.put(target, data=body, timeout=(30, 300))
            print()
            if response.status_code < 500:
                check(response)
                remote_checksum = response.json().get("checksum", "")
                if remote_checksum != f"md5:{local_md5}":
                    print(f"  checksum mismatch: local md5:{local_md5}, "
                          f"remote {remote_checksum}", file=sys.stderr)
                    return False
                print(f"  done, checksum verified ({local_md5})")
                return True
            print(f"  server error {response.status_code}", file=sys.stderr)
        except requests.RequestException as error:
            print(f"\n  network error: {error}", file=sys.stderr)
        except KeyboardInterrupt:
            body.close()
            raise
        finally:
            body.close()

        if attempt < MAX_ATTEMPTS:
            # Jitter keeps repeated failures from lining up on the same
            # retry instant, which matters when the link drops periodically.
            backoff = min(5 * 2 ** (attempt - 1), BACKOFF_CAP)
            backoff += random.uniform(0, backoff / 4)
            print(f"  retrying in {backoff:.0f}s ...", file=sys.stderr)
            time.sleep(backoff)

    print(f"  giving up on {filename} for now", file=sys.stderr)
    return False


already_there = remote_entries(deposition_id)
print(f"Draft currently holds {len(already_there)} file(s): "
      f"{', '.join(sorted(already_there)) or '(none)'}")

succeeded, failed = [], []
try:
    for path in FILES:
        if upload_file(path, already_there):
            succeeded.append(os.path.basename(path))
        else:
            failed.append(os.path.basename(path))
except KeyboardInterrupt:
    print("\n\nInterrupted. Files already finished stay in the draft; "
          "rerun this cell to pick up the rest.")

print(f"\n{'-' * 70}")
print(f"complete: {len(succeeded)}/{len(FILES)}")
if failed:
    print(f"still missing: {', '.join(failed)}")
    print("Rerun this cell to retry only those; the finished ones are skipped.")
else:
    print("All files are in the draft.")


In [ ]:
# STEP 3: attach the metadata

if SET_METADATA:
    print("\nSetting metadata (replaces all existing metadata fields) ...")
    check(session.put(f"{BASE_URL}/deposit/depositions/{deposition_id}",
                      json=METADATA, timeout=30))
else:
    print("\nLeaving metadata untouched (SET_METADATA = False).")


In [ ]:
# STEP 4: publish, or stop at the draft

if PUBLISH:
    print("Publishing ...")
    response = check(session.post(
        f"{BASE_URL}/deposit/depositions/{deposition_id}/actions/publish",
        timeout=120))
    record = response.json()
    print(f"\nPublished: {record.get('doi')}")
    print(record["links"].get("record_html", record["links"].get("html")))
else:
    print("\nDraft ready, not published (PUBLISH = False).")
    print(f"Review and publish here: {deposition['links']['html']}")